# Analisis Kesalahan ConvNeXtV2

Notebook ini menampilkan gambar test yang label aslinya `0`, tetapi diprediksi sebagai `2`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

DATA_DIR = Path('/kaggle/input/datasets/<username-kaggle>/satriangawihitam/dataset')
GROUND_TRUTH_PATH = Path('/kaggle/input/datasets/<username-kaggle>/satriangawihitam/submission_freematch_confidence.csv')
PREDICTION_PATH = Path('/kaggle/working/base_models/convnextv2_base/submission.csv')

TEST_DIR = DATA_DIR / 'test'
OUTPUT_PATH = Path('/kaggle/working/errors_true0_pred2.csv')


In [ ]:
assert GROUND_TRUTH_PATH.exists(), f'Ground truth tidak ditemukan: {GROUND_TRUTH_PATH}'
assert PREDICTION_PATH.exists(), f'Hasil model tidak ditemukan: {PREDICTION_PATH}'
assert TEST_DIR.exists(), f'Folder test tidak ditemukan: {TEST_DIR}'

ground_truth = pd.read_csv(GROUND_TRUTH_PATH, dtype={'id': str})[['id', 'predicted']].rename(
    columns={'predicted': 'true_label'}
)
prediction = pd.read_csv(PREDICTION_PATH, dtype={'id': str})[['id', 'predicted']].rename(
    columns={'predicted': 'pred_label'}
)

comparison = ground_truth.merge(prediction, on='id', how='inner', validate='one_to_one')
comparison[['true_label', 'pred_label']] = comparison[['true_label', 'pred_label']].astype(int)

errors_0_to_2 = comparison.query('true_label == 0 and pred_label == 2').copy()
errors_0_to_2.to_csv(OUTPUT_PATH, index=False)

print(f'Jumlah data dibandingkan: {len(comparison)}')
print(f'Kesalahan true=0 -> pred=2: {len(errors_0_to_2)}')
print(f'CSV kesalahan disimpan di: {OUTPUT_PATH}')
errors_0_to_2.head()

In [ ]:
n_cols = 5
n_rows = (len(errors_0_to_2) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 3.8 * n_rows))
axes = axes.ravel()

for ax, row in zip(axes, errors_0_to_2.itertuples(index=False)):
    image_path = TEST_DIR / f'{row.id}.jpg'
    with Image.open(image_path) as image:
        ax.imshow(image.convert('RGB'))
    ax.set_title(f'id={row.id} | true=0, pred=2', fontsize=9)
    ax.axis('off')

for ax in axes[len(errors_0_to_2):]:
    ax.axis('off')

fig.suptitle('35 kesalahan: label asli 0, prediksi 2', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()